## EDA marathos - silver 
- cleaning


In [0]:
df = spark.sql("FROM marathos.bronze.raw_marathos")
df.display()

In [0]:
df.columns

In [0]:
import re 

#tested to do both functions but will use snake_case for good readibility 

def to_snake_case(name):
    snakecase = snakecase =re.sub(r"[\s]+", "_", name.strip().lower())
    return snakecase

def to_camelCase(name): 
    snakecase = to_snake_case(name)
    words = snakecase.split("_")
    camelcase = words[0].lower() + "".join(word.capitalize() for word in words[1:])
    return camelcase


In [0]:
to_snake_case("Athlete club")

In [0]:
to_camelCase("Athlete club")

In [0]:
def rename_columns_to_snakecase(df): 
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns) 

In [0]:
df_cleaned_columns = rename_columns_to_snakecase(df)
df_cleaned_columns.display()

In [0]:
for column in df.columns: 
    print(column, df.filter(df[column].isNull()).count())

In [0]:
from pyspark.sql.functions import to_timestamp, col, coalesce, lit, when, round as spark_round

In [0]:
df_cleaned_columns.select("event_distance/length", "athlete_performance").limit(30).display()

In [0]:
df_cleaned_columns.select("athlete_performance","athlete_average_speed","event_number_of_finishers").orderBy("athlete_average_speed").display()


In [0]:
def clean_null_values_string(df):
    for column, col_type in df.dtypes: 
        if col_type == "string": 
            df = df.withColumn(column, coalesce(col(column), lit("unknown")))
    return df

df_cleaned = clean_null_values(df_cleaned_columns)
df_cleaned.display()

In [0]:
# Year of event 0
# Event dates 0
# Event name 0
# Event distance/length 0
# Event number of finishers 0
# Athlete performance 2
# Athlete club 2826373
# Athlete country 3
# Athlete year of birth 588161
# Athlete gender 7
# Athlete age category 584938
# Athlete average speed 224
# Athlete ID 0

# Year of event:integer
# Event dates:string
# Event name:string
# Event distance/length:string
# Event number of finishers:integer
# Athlete performance:string
# Athlete club:string
# Athlete country:string
# Athlete year of birth:double
# Athlete gender:string
# Athlete age category:string
# Athlete average speed:string
# Athlete ID:integer

In [0]:
# Cleaned dataframe
df_cleaned = (
    df_cleaned_columns.withColumn(
        "athlete_year_of_birth",
        coalesce(col("athlete_year_of_birth").cast("int").cast("string"), lit("unknown")),
    )
    .withColumn(
        "event_unit",
        when(col("event_distance/length").contains("km"), "km")
        .when(col("event_distance/length").contains("mi"), "mi")
        .when(col("event_distance/length").contains("h"), "h")
        .otherwise("unknown"),
    )
    .withColumn(
        "performance_unit",
        when(col("athlete_performance").contains("km"), "km")
        .when(col("athlete_performance").contains("h"), "h")
        .otherwise("unknown"),
    )
)

df_cleaned.select(
    "event_distance/length", "event_unit", "athlete_performance", "performance_unit","athlete_year_of_birth"
).display()